In [1]:
# -------------------------------------------------------------------------
# 0. Clean & Safe Dependencies Installation
# -------------------------------------------------------------------------
!pip install -q transformers peft bitsandbytes accelerate scikit-learn pandas numpy tqdm



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 42.1 MB/s eta 0:00:00


In [2]:
# -------------------------------------------------------------------------
# 1. Model 4 Training: Qwen-2.5-3B + RAD-DINO + QLoRA + RAG + Multi-Session Resume
# -------------------------------------------------------------------------
import os, sys, gc, glob, time, json, math, random, torch
from pathlib import Path
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    get_cosine_schedule_with_warmup
)
from peft import (
    LoraConfig, get_peft_model, prepare_model_for_kbit_training,
    set_peft_model_state_dict, get_peft_model_state_dict
)
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

# Set seed for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training Compute Device: {device}", flush=True)

work_dir = "/kaggle/working"
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

# -------------------------------------------------------------------------
# Auto-Locate Data Prep Artifacts (from Notebook 1 via + Add Input)
# -------------------------------------------------------------------------
search_dirs = [Path("/kaggle/working"), Path("/kaggle/input")]
manifest_path = None
features_base = None

for base in search_dirs:
    if not base.exists():
        continue
    for root, _, files in os.walk(base):
        if "mimic_master_split_with_rag.csv" in files and manifest_path is None:
            manifest_path = os.path.join(root, "mimic_master_split_with_rag.csv")
            features_base = root
            break
    if manifest_path:
        break

if not manifest_path:
    raise FileNotFoundError(
        "Missing mimic_master_split_with_rag.csv! Please add Notebook 1's published dataset "
        "via '+ Add Input' -> 'Your Work' / 'Datasets'."
    )

print(f"Found Master Manifest : {manifest_path}", flush=True)
print(f"Found Features Directory: {features_base}", flush=True)

df_merged = pd.read_csv(manifest_path)
train_df = df_merged[df_merged["split"] == "train"].reset_index(drop=True)
val_df = df_merged[df_merged["split"] == "val"].reset_index(drop=True)
test_df = df_merged[df_merged["split"] == "test"].reset_index(drop=True)

CHEXPERT_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Enlarged Cardiomediastinum",
    "Fracture", "Lung Lesion", "Lung Opacity", "No Finding", "Pleural Effusion",
    "Pleural Other", "Pneumonia", "Pneumothorax", "Support Devices"
]

# Random sampling of 15,000 diverse training cases while strictly preserving orig_split_idx
max_samples = min(15000, len(train_df))
if max_samples < len(train_df):
    train_sampled_df = train_df.sample(n=max_samples, random_state=42).reset_index(drop=True)
else:
    train_sampled_df = train_df.copy()

print(f"Dataset Cohort: Total Train={len(train_df)} (Sampled={len(train_sampled_df)}) | Val={len(val_df)} | Test={len(test_df)}", flush=True)

# -------------------------------------------------------------------------
# Qwen-2.5-3B Architecture & LoRA Configuration (r=32, alpha=64)
# -------------------------------------------------------------------------
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_qwen = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
    trust_remote_code=True
)

base_qwen = prepare_model_for_kbit_training(base_qwen)
base_qwen.config.use_cache = False
base_qwen.gradient_checkpointing_enable()

# Upgraded LoRA rank: r=32, alpha=64 as specified in prompt
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
qwen_lora_model = get_peft_model(base_qwen, peft_config)

class MultiTaskQwenModel(nn.Module):
    def __init__(self, qwen_model, visual_dim=768, qwen_dim=2048, num_patches=128):
        super().__init__()
        self.qwen = qwen_model
        self.num_patches = num_patches
        self.qwen_dim = qwen_dim
        
        self.visual_projection = nn.Sequential(
            nn.Linear(visual_dim, qwen_dim),
            nn.GELU(),
            nn.Linear(qwen_dim, qwen_dim)
        ).to(device=device)
        
        self.classifier = nn.Sequential(
            nn.Linear(visual_dim, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, 14)
        ).to(device=device)
        
    def build_4d_prefix_attention_mask(self, batch_size, seq_len, device, dtype, text_pad_mask=None):
        total_len = self.num_patches + seq_len
        mask = torch.full((total_len, total_len), -10000.0, dtype=dtype, device=device)
        mask[:self.num_patches, :self.num_patches] = 0.0
        for i in range(self.num_patches, total_len):
            mask[i, :i+1] = 0.0
            
        mask_4d = mask.unsqueeze(0).unsqueeze(0).expand(batch_size, 1, total_len, total_len).clone()
        if text_pad_mask is not None:
            pad_expanded = text_pad_mask.unsqueeze(1).unsqueeze(2)
            mask_4d[:, :, :, self.num_patches:] = mask_4d[:, :, :, self.num_patches:].masked_fill(pad_expanded == 0, -10000.0)
        return mask_4d
        
    def forward(self, spatial_tokens, mean_features, input_ids, labels=None, text_pad_mask=None):
        batch_size = spatial_tokens.size(0)
        seq_len = input_ids.size(1)
        
        cls_logits = self.classifier(mean_features.float())
        visual_embeds = self.visual_projection(spatial_tokens.float())
        text_embeds = self.qwen.get_input_embeddings()(input_ids)
        
        if visual_embeds.dtype != text_embeds.dtype:
            visual_embeds = visual_embeds.to(text_embeds.dtype)
            
        inputs_embeds = torch.cat([visual_embeds, text_embeds], dim=1)
        attention_mask_4d = self.build_4d_prefix_attention_mask(batch_size, seq_len, spatial_tokens.device, inputs_embeds.dtype, text_pad_mask)
        
        if labels is not None:
            vis_labels = torch.full((batch_size, self.num_patches), -100, dtype=labels.dtype, device=labels.device)
            combined_labels = torch.cat([vis_labels, labels], dim=1)
        else:
            combined_labels = None
            
        outputs = self.qwen(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask_4d,
            labels=combined_labels
        )
        return outputs.loss, cls_logits

model = MultiTaskQwenModel(qwen_lora_model).to(device)

# -------------------------------------------------------------------------
# Dataset & DataLoader (Shape-Safe Memory Mapping)
# -------------------------------------------------------------------------
def find_artifact_file(target_filename, fallback_folder):
    for s_dir in search_dirs:
        if not s_dir.exists(): continue
        for r, _, f_list in os.walk(s_dir):
            if target_filename in f_list:
                return os.path.join(r, target_filename)
    return os.path.join(fallback_folder, target_filename)

class FeaturesDataset(Dataset):
    def __init__(self, df, split_name, total_split_len, tokenizer, max_seq_len=160, max_rag_tokens=40):
        self.df = df
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.max_rag_tokens = max_rag_tokens
        
        spatial_path = find_artifact_file(f"features_spatial_{split_name}.dat", features_base)
        mean_path = find_artifact_file(f"features_mean_{split_name}.dat", features_base)
        
        self.spatial_mmap = np.memmap(spatial_path, dtype="float16", mode="r", shape=(total_split_len, 128, 768))
        self.mean_mmap = np.memmap(mean_path, dtype="float32", mode="r", shape=(total_split_len, 768))
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        orig_idx = int(row["orig_split_idx"])
        spatial_token = torch.from_numpy(np.array(self.spatial_mmap[orig_idx]))
        mean_feat = torch.from_numpy(np.array(self.mean_mmap[orig_idx]))
        cls_labels = torch.tensor([row[c] for c in CHEXPERT_COLS], dtype=torch.float)
        
        retrieved_raw = row["retrieved_report"]
        if pd.isna(retrieved_raw) or str(retrieved_raw).strip().lower() in ["", "nan", "none"]:
            retrieved_clean = "Normal chest radiograph with no acute cardiopulmonary abnormality."
        else:
            retrieved_clean = str(retrieved_raw).strip()
            
        # Clean RAG truncation
        rag_tokens = self.tokenizer.encode(retrieved_clean, add_special_tokens=False)[:self.max_rag_tokens]
        retrieved_clean = self.tokenizer.decode(rag_tokens, skip_special_tokens=True)
            
        gt_report = row["report"] if pd.notna(row["report"]) else "Normal chest radiograph."
        
        messages = [
            {"role": "system", "content": "You are a professional radiologist assistant. Generate a concise, diagnostic chest X-ray findings impression report based on the image features and reference exemplar."},
            {"role": "user", "content": f"Reference Report:\n{retrieved_clean}\n\nGenerate the chest X-ray findings report:"},
            {"role": "assistant", "content": gt_report}
        ]
        prompt_str = self.tokenizer.apply_chat_template(messages, tokenize=False)
        
        user_only_msgs = messages[:2]
        user_str = self.tokenizer.apply_chat_template(user_only_msgs, tokenize=False, add_generation_prompt=True)
        user_tokens = self.tokenizer.encode(user_str, add_special_tokens=False)
        
        enc = self.tokenizer(
            prompt_str,
            truncation=True,
            max_length=self.max_seq_len,
            padding="max_length",
            return_tensors="pt"
        )
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        user_len = min(len(user_tokens), self.max_seq_len)
        labels[:user_len] = -100
        
        return spatial_token, mean_feat, input_ids, labels, attention_mask, cls_labels

train_ds = FeaturesDataset(train_sampled_df, "train", len(train_df), tokenizer)
val_ds = FeaturesDataset(val_df, "val", len(val_df), tokenizer)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0, pin_memory=False)

# -------------------------------------------------------------------------
# Focal Loss (Rare Pathology Handling)
# -------------------------------------------------------------------------
pos_counts = train_df[CHEXPERT_COLS].sum().values
neg_counts = len(train_df) - pos_counts
pos_weights = np.clip(neg_counts / np.maximum(pos_counts, 1.0), 1.0, 10.0)
pos_weights_tensor = torch.tensor(pos_weights, dtype=torch.float, device=device)

class FocalLoss(nn.Module):
    def __init__(self, pos_weight, gamma=2.0):
        super().__init__()
        self.pos_weight = pos_weight
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction="none"
        )
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        focal_weight = (1 - p_t) ** self.gamma
        return (focal_weight * bce).mean()

criterion_cls = FocalLoss(pos_weight=pos_weights_tensor, gamma=2.0)

# -------------------------------------------------------------------------
# Optimizer & Learning Rate Scheduling (visual_projection lr=3e-4)
# -------------------------------------------------------------------------
trainable_params = [
    {"params": model.visual_projection.parameters(), "lr": 3e-4, "weight_decay": 0.01},
    {"params": model.classifier.parameters(), "lr": 1e-4, "weight_decay": 0.01},
    {"params": [p for p in model.qwen.parameters() if p.requires_grad], "lr": 5e-5, "weight_decay": 0.01}
]
optimizer = torch.optim.AdamW(trainable_params)

# Global horizon: 4 total epochs planned across sessions
GLOBAL_TOTAL_EPOCHS = 4
EPOCHS_PER_SESSION = 2
accumulation_steps = 4
steps_per_epoch = len(train_loader) // accumulation_steps
total_training_steps = steps_per_epoch * GLOBAL_TOTAL_EPOCHS

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.05 * total_training_steps),
    num_training_steps=total_training_steps
)

# -------------------------------------------------------------------------
# Multi-Session Resume Logic
# -------------------------------------------------------------------------
start_epoch = 1
best_val_loss = float("inf")

# Search for existing checkpoint in /kaggle/input (from previous run) or /kaggle/working
prev_ckpts = glob.glob("/kaggle/input/**/model4_qwen_latest.pt", recursive=True) + \
             glob.glob("/kaggle/input/**/model4_qwen_best.pt", recursive=True) + \
             glob.glob("/kaggle/working/model4_qwen_latest.pt") + \
             glob.glob("/kaggle/working/model4_qwen_best.pt")

resume_ckpt_path = prev_ckpts[0] if prev_ckpts else None

if resume_ckpt_path and os.path.exists(resume_ckpt_path):
    print(f"Resuming training from checkpoint: {resume_ckpt_path}", flush=True)
    ckpt = torch.load(resume_ckpt_path, map_location=device)
    model.visual_projection.load_state_dict(ckpt["visual_projection"])
    model.classifier.load_state_dict(ckpt["classifier"])
    set_peft_model_state_dict(qwen_lora_model, ckpt["lora_state"])
    if "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    if "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])
    start_epoch = ckpt.get("epoch", 1) + 1
    best_val_loss = ckpt.get("val_loss", float("inf"))
    print(f"Resumed state loaded! Starting at Epoch {start_epoch} (Prior Best Val Loss: {best_val_loss:.4f})", flush=True)
else:
    print("No prior checkpoint found. Initializing training from scratch (Epoch 1).", flush=True)

end_epoch = start_epoch + EPOCHS_PER_SESSION - 1
print(f"Session Execution Target: Epoch {start_epoch} to Epoch {end_epoch} (Max Global: {GLOBAL_TOTAL_EPOCHS})", flush=True)

# -------------------------------------------------------------------------
# Training Loop
# -------------------------------------------------------------------------
scaler = torch.amp.GradScaler("cuda")
training_history = []
prior_history_files = glob.glob("/kaggle/input/**/training_history.json", recursive=True)
if prior_history_files and os.path.exists(prior_history_files[0]):
    try:
        with open(prior_history_files[0]) as f:
            training_history = json.load(f)
        print(f"Loaded {len(training_history)} prior epoch records from previous session training_history.json", flush=True)
    except Exception:
        pass

for epoch in range(start_epoch, end_epoch + 1):
    epoch_start_time = time.time()
    model.train()
    running_joint_loss, running_text_loss, running_cls_loss = 0.0, 0.0, 0.0
    optimizer.zero_grad()
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{end_epoch} [Train]")
    for step, (spatial_t, mean_f, input_ids, labels, attn_mask, cls_targets) in enumerate(pbar):
        spatial_t = spatial_t.to(device)
        mean_f = mean_f.to(device)
        input_ids = input_ids.to(device)
        labels = labels.to(device)
        attn_mask = attn_mask.to(device)
        cls_targets = cls_targets.to(device)
        
        with torch.amp.autocast("cuda"):
            loss_text, cls_logits = model(spatial_t, mean_f, input_ids, labels=labels, text_pad_mask=attn_mask)
            loss_cls = criterion_cls(cls_logits, cls_targets)
            joint_loss = (loss_text + 1.0 * loss_cls) / accumulation_steps
            
        scaler.scale(joint_loss).backward()
        
        if (step + 1) % accumulation_steps == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            all_trainable = [p for p in model.parameters() if p.requires_grad]
            torch.nn.utils.clip_grad_norm_(all_trainable, max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            
        running_joint_loss += joint_loss.item() * accumulation_steps
        running_text_loss += loss_text.item()
        running_cls_loss += loss_cls.item()
        
        if (step + 1) % 10 == 0 or (step + 1) == len(train_loader):
            current_lr = scheduler.get_last_lr()[0]
            pbar.set_postfix({
                "Joint": f"{joint_loss.item() * accumulation_steps:.4f}",
                "Text": f"{loss_text.item():.4f}",
                "Cls": f"{loss_cls.item():.4f}",
                "LR": f"{current_lr:.2e}"
            })
            
    epoch_train_joint = running_joint_loss / len(train_loader)
    epoch_train_text = running_text_loss / len(train_loader)
    epoch_train_cls = running_cls_loss / len(train_loader)
    
    # ---------------------------------------------------------------------
    # Validation Phase
    # ---------------------------------------------------------------------
    model.eval()
    val_joint_loss, val_text_loss, val_cls_loss = 0.0, 0.0, 0.0
    all_val_probs, all_val_targets = [], []
    
    with torch.no_grad():
        for spatial_t, mean_f, input_ids, labels, attn_mask, cls_targets in tqdm(val_loader, desc=f"Epoch {epoch}/{end_epoch} [Val]"):
            spatial_t = spatial_t.to(device)
            mean_f = mean_f.to(device)
            input_ids = input_ids.to(device)
            labels = labels.to(device)
            attn_mask = attn_mask.to(device)
            cls_targets = cls_targets.to(device)
            
            with torch.amp.autocast("cuda"):
                loss_text, cls_logits = model(spatial_t, mean_f, input_ids, labels=labels, text_pad_mask=attn_mask)
                loss_cls = criterion_cls(cls_logits, cls_targets)
                joint_loss = loss_text + 1.0 * loss_cls
                
            val_joint_loss += joint_loss.item()
            val_text_loss += loss_text.item()
            val_cls_loss += loss_cls.item()
            
            all_val_probs.append(torch.sigmoid(cls_logits).cpu().numpy())
            all_val_targets.append(cls_targets.cpu().numpy())
            
    epoch_val_joint = val_joint_loss / len(val_loader)
    epoch_val_text = val_text_loss / len(val_loader)
    epoch_val_cls = val_cls_loss / len(val_loader)
    epoch_duration = (time.time() - epoch_start_time) / 60.0
    
    print(f"\n{'='*75}", flush=True)
    print(f"Epoch {epoch}/{end_epoch} Summary ({epoch_duration:.1f} mins):", flush=True)
    print(f"  Train: Joint={epoch_train_joint:.4f} | Text={epoch_train_text:.4f} | Cls={epoch_train_cls:.4f}", flush=True)
    print(f"  Val  : Joint={epoch_val_joint:.4f} | Text={epoch_val_text:.4f} | Cls={epoch_val_cls:.4f}", flush=True)
    print(f"{'='*75}\n", flush=True)
    
    # Save checkpoint state dict
    ckpt_state = {
        "visual_projection": model.visual_projection.state_dict(),
        "classifier": model.classifier.state_dict(),
        "lora_state": {k: v.cpu() for k, v in get_peft_model_state_dict(qwen_lora_model).items()},
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "epoch": epoch,
        "val_loss": epoch_val_joint
    }
    
    # Always save latest checkpoint
    torch.save(ckpt_state, "/kaggle/working/model4_qwen_latest.pt")
    print(f"Saved latest checkpoint to /kaggle/working/model4_qwen_latest.pt", flush=True)
    
    # Save best checkpoint
    if epoch_val_joint < best_val_loss:
        best_val_loss = epoch_val_joint
        torch.save(ckpt_state, "/kaggle/working/model4_qwen_best.pt")
        print(f"[*] New best validation loss achieved ({best_val_loss:.4f})! Saved model4_qwen_best.pt", flush=True)
        
    training_history.append({
        "epoch": epoch,
        "train_joint": epoch_train_joint, "train_text": epoch_train_text, "train_cls": epoch_train_cls,
        "val_joint": epoch_val_joint, "val_text": epoch_val_text, "val_cls": epoch_val_cls
    })

with open("/kaggle/working/training_history.json", "w") as f:
    json.dump(training_history, f, indent=2)

# Generate and save Learning Curves
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = [h["epoch"] for h in training_history]
ax1.plot(epochs_range, [h["train_joint"] for h in training_history], marker="o", label="Train Joint Loss")
ax1.plot(epochs_range, [h["val_joint"] for h in training_history], marker="s", label="Val Joint Loss")
ax1.set_title("Model 4 Joint Loss Progression", fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.6)

ax2.plot(epochs_range, [h["train_text"] for h in training_history], marker="o", label="Train Text Loss")
ax2.plot(epochs_range, [h["val_text"] for h in training_history], marker="s", label="Val Text Loss")
ax2.set_title("Model 4 Text Causal Loss Progression", fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig("/kaggle/working/learning_curves.png", dpi=300)
plt.close()
print("Learning curves plot saved to /kaggle/working/learning_curves.png", flush=True)

# -------------------------------------------------------------------------
# Step: Compute Optimal Class Thresholds on Validation Set
# -------------------------------------------------------------------------
print("\nComputing Optimal Pathology Thresholds on Validation Cohort...", flush=True)
val_probs_arr = np.concatenate(all_val_probs, axis=0)
val_targets_arr = np.concatenate(all_val_targets, axis=0)

optimal_thresholds = {}
for i, col in enumerate(CHEXPERT_COLS):
    best_f1, best_thresh = 0.0, 0.5
    for thresh in np.arange(0.1, 0.9, 0.02):
        preds = (val_probs_arr[:, i] >= thresh).astype(int)
        f1 = f1_score(val_targets_arr[:, i], preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh
    optimal_thresholds[col] = float(np.round(best_thresh, 3))
    print(f"  {col:<26} | Optimal Thresh: {best_thresh:.2f} | Val F1: {best_f1:.4f}", flush=True)

with open("/kaggle/working/optimal_thresholds.json", "w") as f:
    json.dump(optimal_thresholds, f, indent=2)

print("\nOptimal Thresholds successfully saved to /kaggle/working/optimal_thresholds.json", flush=True)
print("Training Session Completed Successfully!", flush=True)



Training Compute Device: cuda
Found Master Manifest : /kaggle/input/datasets/mikuchinu/mimic-cxr-model4-prep-data/mimic_master_split_with_rag.csv
Found Features Directory: /kaggle/input/datasets/mikuchinu/mimic-cxr-model4-prep-data
Dataset Cohort: Total Train=50176 (Sampled=15000) | Val=6190 | Test=6388


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (8 outputs left out)


Epoch 3/4 Summary (170.4 mins):
  Train: Joint=1.4544 | Text=1.2919 | Cls=0.1625
  Val  : Joint=1.6455 | Text=1.4665 | Cls=0.1790

Saved latest checkpoint to /kaggle/working/model4_qwen_latest.pt
[*] New best validation loss achieved (1.6455)! Saved model4_qwen_best.pt


Epoch 4/4 [Train]:   0%|          | 0/1875 [00:00<?, ?it/s]

Epoch 4/4 [Val]:   0%|          | 0/774 [00:00<?, ?it/s]


Epoch 4/4 Summary (168.9 mins):
  Train: Joint=1.3363 | Text=1.1759 | Cls=0.1604
  Val  : Joint=1.6669 | Text=1.4879 | Cls=0.1790

Saved latest checkpoint to /kaggle/working/model4_qwen_latest.pt
Learning curves plot saved to /kaggle/working/learning_curves.png

Computing Optimal Pathology Thresholds on Validation Cohort...
  Atelectasis                | Optimal Thresh: 0.54 | Val F1: 0.4690
  Cardiomegaly               | Optimal Thresh: 0.54 | Val F1: 0.4570
  Consolidation              | Optimal Thresh: 0.46 | Val F1: 0.1991
  Edema                      | Optimal Thresh: 0.58 | Val F1: 0.4454
  Enlarged Cardiomediastinum | Optimal Thresh: 0.44 | Val F1: 0.1330
  Fracture                   | Optimal Thresh: 0.40 | Val F1: 0.0800
  Lung Lesion                | Optimal Thresh: 0.48 | Val F1: 0.1605
  Lung Opacity               | Optimal Thresh: 0.52 | Val F1: 0.4334
  No Finding                 | Optimal Thresh: 0.50 | Val F1: 0.7369
  Pleural Effusion           | Optimal Thresh: 0.54 